<a href="https://colab.research.google.com/github/imid12/neuraleagleshcc/blob/main/Sleeeeppp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Core libraries
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Regressors
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.neighbors import KNeighborsRegressor

# Evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
# Load your dataset (update the file name as needed)
df = pd.read_csv("sleep_cycle_productivity.csv")  # Replace with your file name
df.head()


,Date,Person_ID,Age,Gender,Sleep Start Time,Sleep End Time,Total Sleep Hours,Sleep Quality,Exercise (mins/day),Caffeine Intake (mg),Screen Time Before Bed (mins),Work Hours (hrs/day),Productivity Score,Mood Score,Stress Level
0,2024-04-12,1860,32,Other,23.33,4.61,5.28,3,86,87,116,8.808920,8,3,6
1,2024-11-04,1769,41,Female,21.02,2.43,5.41,5,32,21,88,6.329833,10,3,7
2,2024-08-31,2528,20,Male,22.10,3.45,5.35,7,17,88,59,8.506306,10,9,10
3,2024-02-22,8041,37,Other,23.10,6.65,7.55,8,46,34,80,6.070240,8,4,2
4,2024-02-23,4843,46,Other,21.42,4.17,6.75,10,61,269,94,11.374994,8,7,9


In [3]:
# Drop ID and Date columns
df_cleaned = df.drop(['Date', 'Person_ID'], axis=1)

# Convert Sleep Start and End Time to datetime
df_cleaned['Sleep Start Time'] = pd.to_datetime(df_cleaned['Sleep Start Time'])
df_cleaned['Sleep End Time'] = pd.to_datetime(df_cleaned['Sleep End Time'])

# Create a new feature: Sleep Duration in minutes
df_cleaned['Sleep Duration (mins)'] = (df_cleaned['Sleep End Time'] - df_cleaned['Sleep Start Time']).dt.total_seconds() / 60.0

# Drop the original time columns
df_cleaned = df_cleaned.drop(['Sleep Start Time', 'Sleep End Time'], axis=1)

# One-hot encode 'Gender'
df_cleaned = pd.get_dummies(df_cleaned, columns=['Gender'], drop_first=True)

# Drop missing values (if any still exist)
df_cleaned = df_cleaned.dropna()

# Split into features and target
X = df_cleaned.drop('Productivity Score', axis=1)
y = df_cleaned['Productivity Score']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [5]:
# 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)


In [6]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "K-Nearest Neighbors": KNeighborsRegressor()
}

val_results = {}
test_results = {}

# Train and evaluate each model
for name, model in models.items():
    model.fit(X_train, y_train)

    val_preds = model.predict(X_val)
    test_preds = model.predict(X_test)

    val_results[name] = {
        "MAE": mean_absolute_error(y_val, val_preds),
        "MSE": mean_squared_error(y_val, val_preds),
        "R²": r2_score(y_val, val_preds)
    }

    test_results[name] = {
        "MAE": mean_absolute_error(y_test, test_preds),
        "MSE": mean_squared_error(y_test, test_preds),
        "R²": r2_score(y_test, test_preds)
    }


In [7]:
# Select top 3 models based on R² on validation set
top_3_models = sorted(val_results.items(), key=lambda x: x[1]["R²"], reverse=True)[:3]
ensemble_estimators = [(name, models[name]) for name, _ in top_3_models]

# Create Voting Regressor
voting_reg = VotingRegressor(ensemble_estimators)
voting_reg.fit(X_train, y_train)

# Predict
val_preds = voting_reg.predict(X_val)
test_preds = voting_reg.predict(X_test)

# Store ensemble metrics
val_results["Voting Ensemble"] = {
    "MAE": mean_absolute_error(y_val, val_preds),
    "MSE": mean_squared_error(y_val, val_preds),
    "R²": r2_score(y_val, val_preds)
}

test_results["Voting Ensemble"] = {
    "MAE": mean_absolute_error(y_test, test_preds),
    "MSE": mean_squared_error(y_test, test_preds),
    "R²": r2_score(y_test, test_preds)
}


In [8]:
val_df = pd.DataFrame(val_results).T
test_df = pd.DataFrame(test_results).T

print("Validation Metrics:")
display(val_df)

print("\nTest Metrics:")
display(test_df)


Validation Metrics:


,MAE,MSE,R²
Linear Regression,2.557984,8.434135,-0.001313
Decision Tree,3.326667,17.201333,-1.042167
Random Forest,2.578320,8.714695,-0.034621
Gradient Boosting,2.566205,8.597263,-0.020680
K-Nearest Neighbors,2.683200,9.903680,-0.175779
Voting Ensemble,2.562565,8.504434,-0.009659



Test Metrics:


,MAE,MSE,R²
Linear Regression,2.501762,8.196473,-0.001071
Decision Tree,3.234667,16.314667,-0.992581
Random Forest,2.518187,8.445323,-0.031464
Gradient Boosting,2.517950,8.378991,-0.023363
K-Nearest Neighbors,2.708533,10.237280,-0.250324
Voting Ensemble,2.507156,8.259318,-0.008746


In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor


In [10]:
# Initialize models
lr = LinearRegression()
dt = DecisionTreeRegressor(random_state=42)
rf = RandomForestRegressor(random_state=42)
xgb = XGBRegressor(random_state=42, verbosity=0)
knn = KNeighborsRegressor()

# Train on the training data
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)
knn.fit(X_train, y_train)


KNeighborsRegressor()

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd


In [15]:
# Function to evaluate a model and return metrics
def evaluate_model(model, X_val, y_val):
    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    return mae, mse, r2

# Evaluate each model
results = {
    "Model": [],
    "MAE": [],
    "MSE": [],
    "R² Score": []
}

models = {
    "Linear Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "XGBoost": xgb,
    "KNN": knn
}

for name, model in models.items():
    mae, mse, r2 = evaluate_model(model, X_val, y_val)
    results["Model"].append(name)
    results["MAE"].append(mae)
    results["MSE"].append(mse)
    results["R² Score"].append(r2)

# Display results in a DataFrame
val_results_df = pd.DataFrame(results)
val_results_df.sort_values("R² Score", ascending=False, inplace=True)
val_results_df


,Model,MAE,MSE,R² Score
0,Linear Regression,2.557984,8.434135,-0.001313
2,Random Forest,2.578320,8.714695,-0.034621
4,KNN,2.683200,9.903680,-0.175779
3,XGBoost,2.691369,10.011315,-0.188558
1,Decision Tree,3.326667,17.201333,-1.042167


In [16]:
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import BayesianRidge


In [17]:
# Create a Voting Regressor with the top 3 models
voting_reg = VotingRegressor(estimators=[
    ('rf', rf),
    ('xgb', xgb),
    ('knn', knn)
])

# Train the Voting Regressor
voting_reg.fit(X_train, y_train)


VotingRegressor(estimators=[('rf', RandomForestRegressor(random_state=42)),
                            ('xgb',
                             XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=None, max_bin=None,
                                          max_cat_threshold=None,
                                          max_cat_to_onehot=None,
                                          max_delta_step=None, max_depth=None,
                                          max_leaves=None,
                                          min_child_weight=None, missing=nan,
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None,
                                          random_state=42, ...)),
                            ('knn', KNeighborsRegressor())])

In [19]:
# Function to evaluate on both sets
def evaluate_ensemble(model, X_val, y_val, X_test, y_test):
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    return {
        "Val MAE": mean_absolute_error(y_val, val_pred),
        "Val MSE": mean_squared_error(y_val, val_pred),
        "Val R²": r2_score(y_val, val_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Test MSE": mean_squared_error(y_test, test_pred),
        "Test R²": r2_score(y_test, test_pred),
    }

voting_results = evaluate_ensemble(voting_reg, X_val, y_val, X_test, y_test)


In [21]:
# Train Bayesian Ridge Regressor
bayesian_model = BayesianRidge()
bayesian_model.fit(X_train, y_train)

# Evaluate
bayesian_results = evaluate_ensemble(bayesian_model, X_val, y_val, X_test, y_test)


In [ ]:
ensemble_df = pd.DataFrame([voting_results, bayesian_results],
                           index=["Voting Regressor", "Bayesian Ridge"])
ensemble_df


,Val MAE,Val MSE,Val R²,Test MAE,Test MSE,Test R²
Voting Regressor,2.590339,8.897031,-0.056269,2.556277,8.836785,-0.079275
Bayesian Ridge,2.561193,8.441337,-0.002168,2.502555,8.195320,-0.000930


In [ ]:
# List of all models to evaluate
final_models = {
    "Linear Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "XGBoost": xgb,
    "KNN": knn,
    "Voting Regressor": voting_reg,
    "Bayesian Ridge": bayesian_model
}

# Function to evaluate on both val and test sets
def evaluate_ensemble(model, X_val, y_val, X_test, y_test):
    val_preds = model.predict(X_val)
    test_preds = model.predict(X_test)
    return {
        "Val MAE": mean_absolute_error(y_val, val_preds),
        "Val MSE": mean_squared_error(y_val, val_preds),
        "Val R²": r2_score(y_val, val_preds),
        "Test MAE": mean_absolute_error(y_test, test_preds),
        "Test MSE": mean_squared_error(y_test, test_preds),
        "Test R²": r2_score(y_test, test_preds)
    }

# Evaluate all models
final_results = []
for name, model in final_models.items():
    metrics = evaluate_ensemble(model, X_val, y_val, X_test, y_test)
    metrics["Model"] = name
    final_results.append(metrics)

# Create final DataFrame
final_comparison_df = pd.DataFrame(final_results)
final_comparison_df = final_comparison_df[[
    "Model", "Val MAE", "Val MSE", "Val R²", "Test MAE", "Test MSE", "Test R²"
]]
final_comparison_df.sort_values("Val R²", ascending=False, inplace=True)
final_comparison_df


,Model,Val MAE,Val MSE,Val R²,Test MAE,Test MSE,Test R²
0,Linear Regression,2.557984,8.434135,-0.001313,2.501762,8.196473,-0.001071
6,Bayesian Ridge,2.561193,8.441337,-0.002168,2.502555,8.195320,-0.000930
2,Random Forest,2.578320,8.714695,-0.034621,2.518187,8.445323,-0.031464
5,Voting Regressor,2.590339,8.897031,-0.056269,2.556277,8.836785,-0.079275
4,KNN,2.683200,9.903680,-0.175779,2.708533,10.237280,-0.250324
3,XGBoost,2.691369,10.011315,-0.188558,2.645081,9.769135,-0.193147
1,Decision Tree,3.326667,17.201333,-1.042167,3.234667,16.314667,-0.992581


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# For Linear Regression
y_val_pred_lr = lr.predict(X_val)
y_test_pred_lr = lr.predict(X_test)

mae_val_lr = mean_absolute_error(y_val, y_val_pred_lr)
mse_val_lr = mean_squared_error(y_val, y_val_pred_lr)
r2_val_lr = r2_score(y_val, y_val_pred_lr)

mae_test_lr = mean_absolute_error(y_test, y_test_pred_lr)
mse_test_lr = mean_squared_error(y_test, y_test_pred_lr)
r2_test_lr = r2_score(y_test, y_test_pred_lr)


In [ ]:
from sklearn.linear_model import BayesianRidge

# Train Bayesian Ridge model
bayes_ridge = BayesianRidge()
bayes_ridge.fit(X_train, y_train)


BayesianRidge()

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# (1) Copy your cleaned DataFrame
df_cls = df_cleaned.copy()

# (2) Bin into three classes
df_cls['Prod_Class'] = pd.qcut(
    df_cls['Productivity Score'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

# (3) Encode the class labels as integers
le = LabelEncoder()
df_cls['Prod_Class_Code'] = le.fit_transform(df_cls['Prod_Class'])
#   Now 'Low'→0, 'Medium'→1, 'High'→2

# (4) Prepare features and numeric target
X_cls = df_cls.drop(['Productivity Score', 'Prod_Class', 'Prod_Class_Code'], axis=1)
y_cls = df_cls['Prod_Class_Code']

# (5) Scale features
scaler = StandardScaler()
X_scaled_cls = scaler.fit_transform(X_cls)

# (6) Split with stratification
X_train_c, X_temp_c, y_train_c, y_temp_c = train_test_split(
    X_scaled_cls, y_cls, test_size=0.30, random_state=42, stratify=y_cls
)
X_val_c, X_test_c, y_val_c, y_test_c = train_test_split(
    X_temp_c, y_temp_c, test_size=0.50, random_state=42, stratify=y_temp_c
)


In [ ]:
for name, clf in clf_models.items():
    clf.fit(X_train_c, y_train_c)


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [19:12:02] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Train 6 Classification Models

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Initialize
clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVC': SVC(probability=True, random_state=42)
}

# Train
for name, clf in clf_models.items():
    clf.fit(X_train_c, y_train_c)


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [19:16:17] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# helper to compute metrics (macro-averaged)
def cls_metrics(estimator, X, y):
    y_pred = estimator.predict(X)
    return {
        'Accuracy':       accuracy_score(y, y_pred),
        'Precision (M)':  precision_score(y, y_pred, average='macro'),
        'Recall (M)':     recall_score(y, y_pred, average='macro'),
        'F1 (M)':         f1_score(y, y_pred, average='macro')
    }

# Evaluate all on validation
results_cls = {}
for name, clf in clf_models.items():
    results_cls[name] = cls_metrics(clf, X_val_c, y_val_c)

val_cls_df = pd.DataFrame(results_cls).T
val_cls_df.sort_values('Accuracy', ascending=False, inplace=True)
val_cls_df


,Accuracy,Precision (M),Recall (M),F1 (M)
Logistic Regression,0.390667,0.582871,0.346579,0.255324
Decision Tree,0.354667,0.352510,0.353067,0.352598
SVC,0.338667,0.303001,0.310302,0.272440
XGBoost,0.337333,0.328251,0.328506,0.326815
KNN,0.334667,0.328729,0.325959,0.317558
Random Forest,0.326667,0.302279,0.308431,0.294775


**Voting Ensemble of Top 3 Classifiers**

In [ ]:

from sklearn.ensemble import VotingClassifier

# pick top 3 by validation accuracy
top3 = val_cls_df.index[:3].tolist()
estimators = [(name, clf_models[name]) for name in top3]

voting_clf = VotingClassifier(estimators=estimators, voting='soft')
voting_clf.fit(X_train_c, y_train_c)

# Evaluate ensemble
results_cls['Voting Ensemble'] = cls_metrics(voting_clf, X_val_c, y_val_c)

val_cls_df = pd.DataFrame(results_cls).T
val_cls_df.sort_values('Accuracy', ascending=False, inplace=True)
val_cls_df


,Accuracy,Precision (M),Recall (M),F1 (M)
Logistic Regression,0.390667,0.582871,0.346579,0.255324
Decision Tree,0.354667,0.352510,0.353067,0.352598
Voting Ensemble,0.354667,0.352510,0.353067,0.352598
SVC,0.338667,0.303001,0.310302,0.272440
XGBoost,0.337333,0.328251,0.328506,0.326815
KNN,0.334667,0.328729,0.325959,0.317558
Random Forest,0.326667,0.302279,0.308431,0.294775


**Final Comparison on Validation & Test**

In [ ]:

# Evaluate Voting on Test
test_results_cls = {}
for name, clf in {**clf_models, 'Voting Ensemble': voting_clf}.items():
    test_results_cls[name] = cls_metrics(clf, X_test_c, y_test_c)

test_cls_df = pd.DataFrame(test_results_cls).T
test_cls_df = test_cls_df.loc[val_cls_df.index]  # align order

# Combine into one table
combined_cls = val_cls_df.add_suffix(' (Val)').join(test_cls_df.add_suffix(' (Test)'))
combined_cls


,Accuracy (Val),Precision (M) (Val),Recall (M) (Val),F1 (M) (Val),Accuracy (Test),Precision (M) (Test),Recall (M) (Test),F1 (M) (Test)
Logistic Regression,0.390667,0.582871,0.346579,0.255324,0.388000,0.371154,0.344014,0.259405
Decision Tree,0.354667,0.352510,0.353067,0.352598,0.368000,0.367257,0.365054,0.365360
Voting Ensemble,0.354667,0.352510,0.353067,0.352598,0.368000,0.367257,0.365054,0.365360
SVC,0.338667,0.303001,0.310302,0.272440,0.361333,0.318842,0.330405,0.291841
XGBoost,0.337333,0.328251,0.328506,0.326815,0.377333,0.368918,0.365692,0.363829
KNN,0.334667,0.328729,0.325959,0.317558,0.358667,0.335755,0.346177,0.332148
Random Forest,0.326667,0.302279,0.308431,0.294775,0.361333,0.331286,0.338850,0.321165
